In [35]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score


In [36]:
df = pd.read_parquet('../data/processed/abcd_first_msgs.parquet')
# df_sample = pd.read_json('../raw_data/abcd_sample.json')


In [37]:
df

,split,id,flow,subflow,first_msgs,refund,refund_label,n_words
0,train,3592,product_defect,return_size,"Hi! I need to return an item, can you help me ...",0,1,13
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund,0,0,10
2,train,3695,storewide_query,timing_4,HEY HO! I've got a promo code and I want to kn...,0,0,27
3,train,5798,shipping_issue,manage,"Hello, I would like to change my shipping deat...",0,1,18
4,train,3647,storewide_query,pricing_3,Hello. I have a really cool party coming up. A...,0,0,34
...,...,...,...,...,...,...,...,...
10037,test,1622,order_issue,manage_upgrade,HI I want to upgrade an order I have with you ...,0,0,14
10038,test,3293,manage_account,status_credit_missing,"Hi, I think I am missing a $40 credit from a s...",0,0,13
10039,test,8492,subscription_inquiry,manage_extension,I would like an extension on my subscription.,0,0,8
10040,test,189,storewide_query,pricing_1,Hi! How much do you charge to gift wrap items?,0,0,10


In [38]:

stop_words = set(ENGLISH_STOP_WORDS) - {"no", "not", "nor", "never"}

def clean_message(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    text = text.replace("’", "'")

    text = text.replace("won't", "will not")
    text = text.replace("can't", "can not")
    text = text.replace("n't", " not")


    text = re.sub(r"\bho!", " ", text)

    text = re.sub(
        r"^(hi+|hello+|helo+|hey+|hiya|good morning|"
        r"good afternoon|good evening)\b[\s,.!]*",
        "", text
    )

    text = re.sub(r"[^\w\s]|_", " ", text)

    words = [word for word in text.split() if word not in stop_words]

    return " ".join(words)

df["clean_text"] = df["first_msgs"].apply(clean_message)

display(df[["first_msgs", "clean_text"]].head(10))

,first_msgs,clean_text
0,"Hi! I need to return an item, can you help me ...",need return item help
1,just wanted to check on the status of a refund,just wanted check status refund
2,HEY HO! I've got a promo code and I want to kn...,ve got promo code want know expire d like use ...
3,"Hello, I would like to change my shipping deat...",like change shipping deatails changed recently
4,Hello. I have a really cool party coming up. A...,really cool party coming need new clothes asap...
5,I want an extension for my shopping service. I...,want extension shopping service need time money
6,"Hello, I am trying to check the status of my o...",trying check status order forgot username not ...
7,Hello I am trying to buy a pair of boots but t...,trying buy pair boots not getting added cart
8,I accidently ordered the wrong item and was wo...,accidently ordered wrong item wondering change...
9,I would like to confirm my address is correct ...,like confirm address correct order able


In [39]:
df

,split,id,flow,subflow,first_msgs,refund,refund_label,n_words,clean_text
0,train,3592,product_defect,return_size,"Hi! I need to return an item, can you help me ...",0,1,13,need return item help
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund,0,0,10,just wanted check status refund
2,train,3695,storewide_query,timing_4,HEY HO! I've got a promo code and I want to kn...,0,0,27,ve got promo code want know expire d like use ...
3,train,5798,shipping_issue,manage,"Hello, I would like to change my shipping deat...",0,1,18,like change shipping deatails changed recently
4,train,3647,storewide_query,pricing_3,Hello. I have a really cool party coming up. A...,0,0,34,really cool party coming need new clothes asap...
...,...,...,...,...,...,...,...,...,...
10037,test,1622,order_issue,manage_upgrade,HI I want to upgrade an order I have with you ...,0,0,14,want upgrade order overnight shipping
10038,test,3293,manage_account,status_credit_missing,"Hi, I think I am missing a $40 credit from a s...",0,0,13,think missing 40 credit subscription refund
10039,test,8492,subscription_inquiry,manage_extension,I would like an extension on my subscription.,0,0,8,like extension subscription
10040,test,189,storewide_query,pricing_1,Hi! How much do you charge to gift wrap items?,0,0,10,charge gift wrap items


In [40]:
# REFUND_SUBFLOWS = {
#     "refund_initiate",
#     "return_size",
#     "return_color",
#     "return_stain",
#     "manage_dispute_bill",
#     "bad_price_competitor",
#     "status_mystery_fee",
#     "refund_update",
#     "manage_cancel",
# }

# df["refund_request"] = df["subflow"].isin(REFUND_SUBFLOWS).astype(int)
# print(df["refund_request"].value_counts())

In [41]:
TARGET = "refund_label"
train = df[df["split"] == "train"]
val = df[df["split"] == "dev"]

X_train = train["clean_text"]
y_train = train[TARGET]

X_val = val["clean_text"]
y_val = val[TARGET]

In [42]:
print(y_train.value_counts(normalize=True))

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_val)

print("accuracy:", accuracy_score(y_val, y_pred))
print("precision:", precision_score(y_val, y_pred, zero_division=0))
print("recall:", recall_score(y_val, y_pred))

refund_label
0    0.776948
1    0.223052
Name: proportion, dtype: float64
accuracy: 0.7719123505976095
precision: 0.0
recall: 0.0


In [43]:
models = {
    "count + none": make_pipeline(
        CountVectorizer(),
        LogisticRegression(max_iter=1000)
    ),
    "count + balanced": make_pipeline(
        CountVectorizer(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    ),
    "tfidf + none": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=1000)
    ),
    "tfidf + balanced": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    )
}

In [44]:
# for name, pipe in models.items():
#     pipe.fit(X_train, y_train)
#     print(f"\n=== {name} ===")
#     print(classification_report(y_val, pipe.predict(X_val)))

In [45]:
results = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)

    results.append({
        "model": name,
        "precision": precision_score(y_val, y_pred),
        "recall": recall_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred)
    })
    results_df = pd.DataFrame(results).sort_values("f1", ascending=False)


In [46]:
results_df


,model,precision,recall,f1
0,count + none,0.933014,0.851528,0.890411
3,tfidf + balanced,0.810078,0.912664,0.858316
2,tfidf + none,0.928205,0.790393,0.853774
1,count + balanced,0.803089,0.908297,0.852459


In [47]:
best_name = results_df.iloc[0]["model"]
best_model = models[best_name]

In [48]:
df[df["refund_label"] == 1]["subflow"].unique().tolist()

['return_size',
 'manage',
 'status_quantity',
 'refund_update',
 'refund_initiate',
 'return_color',
 'bad_price_competitor',
 'status_mystery_fee',
 'cost',
 'manage_dispute_bill',
 'status_service_added',
 'manage_cancel',
 'return_stain']

In [54]:
prompt = "I just found a pair of pants for a cheaper price from a competitor."
refund_prob = best_model.predict_proba([prompt])[0][1]
print(f"Refund request: {refund_prob:.0%}")

Refund request: 66%


In [50]:
check_words = ["back", "get", "give", "want", "money", "return", "refund"]
[word for word in check_words if word in stop_words]

['back', 'get', 'give']